# 实验二：房源采集与 MySQL 存储

- 课程：大数据采集与处理　　班级：数据1241　　姓名：鲁嘉睿　　学号：112404260130
- 数据源：安居客教学站 `http://www.bspider.top/anjuke/`
- 采集字段：`title / house / address / struct / area / floor / maketime / price`
- 本 Notebook 按指导书分为三阶段：①请求与解析　②翻页与 CSV　③MySQL 入库

## 第二阶段 1　获取第一页

先请求教学站并检查返回内容。教学站带模拟反爬：仅使用 `Mozilla/5.0` 这种残缺 UA 会被判定为抓取工具，
返回标题为“访问验证-安居客”的拦截页。因此这里补全为完整的 Chrome UA，并在 `fetch()` 中显式检测拦截页。

In [1]:
import csv
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

BASE_URL = "http://www.bspider.top/anjuke/"
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Connection": "keep-alive",
}
TIMEOUT = 30          # 单次请求超时（秒）
SLEEP_SECONDS = 1.0   # 翻页间隔，至少 1 秒
MAX_PAGES = 30        # 安全页数上限，防止无限循环
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / "houseinfo.csv"

print("字段顺序：", FIELDS)

字段顺序： ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']


In [2]:
class AntiSpamError(RuntimeError):
    "教学站返回访问验证页时抛出"


class StructureError(RuntimeError):
    "页面结构与预期不符（房源容器或分页区缺失）时抛出"


def fetch(url, session=None, tries=3, timeout=TIMEOUT):
    "请求页面并做三项检查：HTTP 状态、拦截页、房源容器是否解析得出"
    sess = session or requests
    last = None
    for attempt in range(1, tries + 1):
        try:
            r = sess.get(url, headers=HEADERS, timeout=timeout)
            r.raise_for_status()                 # 4xx/5xx 直接报错
            r.encoding = r.apparent_encoding     # 用页面声明的编码，避免中文乱码
            soup = BeautifulSoup(r.text, "html.parser")
            title = soup.title.get_text(strip=True) if soup.title else ""
            if "访问验证" in title or "访问验证" in r.text[:2000]:
                raise AntiSpamError(f"命中访问验证页：{title}")
            return soup
        except AntiSpamError:
            raise
        except Exception as exc:                 # 网络抖动：重试
            last = exc
            print(f"  [{attempt}/{tries}] 请求失败 {type(exc).__name__}，2 秒后重试")
            time.sleep(2)
    raise RuntimeError(f"请求多次失败：{url} -> {last}")


soup = fetch(BASE_URL)
print("状态码检查：通过（raise_for_status 未抛错）")
print("网页标题：", soup.title.get_text(strip=True))
print("是否拦截页：", "访问验证" in soup.title.get_text(strip=True))

  [1/3] 请求失败 ConnectionError，2 秒后重试


状态码检查：通过（raise_for_status 未抛错）
网页标题： 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
是否拦截页： False


## 第二阶段 2　分析网页，提取 8 个字段

**选择器是怎么得到的**：在 Chrome 中按 F12 → Elements → 选中第一条房源的标题 → 右键
`Copy → Copy selector`，得到的是一串从根节点开始、带 `nth-child()` 的长路径。那种路径只要页面多插入
一个元素就失效，所以改为两步：

1. 先找**恰好包住一条房源**的父容器 —— `div.property`（同时含标题、楼盘、地址、属性和价格）；
2. 再在**这一条内部**用相对选择器取字段 —— `row.select_one("h3.property-content-title-name")`。

下表是 8 个字段最终使用的相对选择器与取值方式。

In [3]:
rows = soup.select("div.property")      # select() 返回列表，一项对应一条房源
print("本页房源条数：", len(rows))
if len(rows) == 0:
    raise StructureError("没有解析到房源容器 div.property，先检查网页或选择器")

row = rows[0]
print("第一条房源的父容器 class：", row.get("class"))
print()
print("—— 父容器内部结构片段（前 900 字符）——")
print(row.prettify()[:900])

本页房源条数： 60
第一条房源的父容器 class： ['property']

—— 父容器内部结构片段（前 900 字符）——
<div class="property" data-v-31e248e6="" data-v-53a259aa="" tongji_tag="fcpc_ersflist_gzcount">
 <a class="property-ex" data-action="esf_list" data-cp='{"broker_id":"101348367"}' data-ep='{"exposure":{"esf_list":1,"broker_id":"101348367"}}' data-lego='{"entity_id":"2115457344527366","tid":"-"}' data-v-53a259aa="" href="https://chongqing.anjuke.com/prop/view/A5887814286?auction=201&amp;hpType=1&amp;entry=102&amp;position=0&amp;kwtype=filter&amp;now_time=1628513553&amp;spread=filtersearch_p&amp;from=from_esf_List_search&amp;index=0" target="_blank">
  <div class="property-image" data-v-53a259aa="">
   <img alt="" class="lazy-img cover" data-v-28819723="" data-v-53a259aa="" src="/static/anjuke/images/120.jpg"/>
   <img class="property-image-vr" data-v-53a259aa="" src="/static/anjuke/images/comm_propdetail_icon_vr_m@2x.png"/>
   <span class="property-image-vr" data-v-53a259aa="" id="main-0">


In [4]:
def text_of(node, default=""):
    "节点存在就取文字，不存在返回默认值（不做任何空格加工，保留原文）"
    if node is None:
        return default
    return node.get_text(strip=True)


def parse_row(row):
    "解析一条房源，返回按 FIELDS 顺序排列的 8 个字段"
    # --- 标题 ---
    title = text_of(row.select_one("h3.property-content-title-name"))

    # --- 楼盘与地址（地址分散在多个 span 中，直接拼接不插空格）---
    house = text_of(row.select_one("p.property-content-info-comm-name"))
    address = text_of(row.select_one("p.property-content-info-comm-address"))

    # --- 户型 ---
    struct = text_of(row.select_one("p.property-content-info-attribute"))

    # --- 属性组：户型 / 面积 / 朝向 / 楼层 / 年代，共 5 项 ---
    # 注意：户型所在的 p 同时带 property-content-info-text，因此它也是第 0 项
    info = [p.get_text(strip=True) for p in row.select("p.property-content-info-text")]
    area = info[1] if len(info) > 1 else ""        # 第 1 项：面积
    floor = info[3] if len(info) > 3 else ""       # 第 3 项：楼层（第 2 项是朝向，不能取错）
    maketime = ""
    if len(info) > 4 and info[4]:                  # 第 4 项：建造年代，缺失时填空字符串
        maketime = info[4] if "建造" in info[4] else ""

    # --- 单价：取 property-price-average（元/㎡），不是 property-price-total（总价/万）---
    price = text_of(row.select_one("p.property-price-average"))

    return {"title": title, "house": house, "address": address, "struct": struct,
            "area": area, "floor": floor, "maketime": maketime, "price": price}


first = parse_row(rows[0])
print("属性组原文（确认顺序：户型/面积/朝向/楼层/年代）：")
print("  ", [p.get_text(strip=True) for p in rows[0].select("p.property-content-info-text")])
print()
print("第一条房源 8 个字段：")
for k in FIELDS:
    print(f"  {k:9s} = {first[k]!r}")

属性组原文（确认顺序：户型/面积/朝向/楼层/年代）：
   ['3室2厅1卫', '89.9㎡', '南北', '低层(共27层)', '2019年建造']

第一条房源 8 个字段：
  title     = '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
  house     = '万科金域学府翰林菁华里'
  address   = '南岸学府大道二塘路59号'
  struct    = '3室2厅1卫'
  area      = '89.9㎡'
  floor     = '低层(共27层)'
  maketime  = '2019年建造'
  price     = '16465元/㎡'


In [5]:
# 再检查第二条和最后一条，确认解析函数对首尾同样成立
for idx in (1, len(rows) - 1):
    rec = parse_row(rows[idx])
    print(f"第 {idx + 1} 条：", [rec[k] for k in FIELDS])
    assert all(isinstance(rec[k], str) for k in FIELDS), "字段必须是字符串"

print()
print("本页面积缺失条数：", sum(1 for r in rows if not parse_row(r)["area"]))
print("本页年代缺失条数：", sum(1 for r in rows if not parse_row(r)["maketime"]))
print("本页单价缺失条数：", sum(1 for r in rows if not parse_row(r)["price"]))

第 2 条： ['黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡']
第 60 条： ['精装修 电梯房 4室2厅 91平', '华润凯旋天地', '南岸弹子石腾龙大道45号', '4室2厅2卫', '91.3㎡', '低层(共32层)', '2015年建造', '15886元/㎡']

本页面积缺失条数： 0
本页年代缺失条数： 0
本页单价缺失条数： 0


## 第二阶段 3　采集下一页

- 下一页节点：`a.next.next-active`；取**属性** `href`（`get_text()` 拿到的是“下一页”文字，不是地址）。
- 补全地址：`href` 只有 `?page=2`，用 `urljoin(当前地址, href)` 拼成绝对地址。
- 停止条件：分页区仍在、但找不到 `a.next.next-active` → 说明已到末页，正常结束；
  分页区直接消失 → 记为结构异常；达到 `MAX_PAGES` → 记为中止，两种情况都不算采集完成。

In [6]:
next_a = soup.select_one("a.next.next-active")
pagination = soup.select_one(".pagination")
print("分页区是否存在：", pagination is not None)
print("下一页节点文字（get_text）：", next_a.get_text(strip=True) if next_a else None)
href = next_a.get("href") if next_a else None
print("下一页 href（get('href')）：", href)
print("urljoin 补全后：", urljoin(BASE_URL, href) if href else None)

分页区是否存在： True
下一页节点文字（get_text）： 下一页
下一页 href（get('href')）： ?page=2
urljoin 补全后： http://www.bspider.top/anjuke/?page=2


In [7]:
def find_next_url(soup_page, current_url):
    "返回下一页绝对地址；到达末页返回 None；分页区缺失视为结构异常"
    if soup_page.select_one(".pagination") is None:
        raise StructureError("分页区域不存在，无法判断翻页状态，记为结构异常")
    a = soup_page.select_one("a.next.next-active")
    if a is None or not a.get("href"):
        return None                      # 分页区在、按钮已禁用 -> 末页
    return urljoin(current_url, a.get("href"))


def crawl(start_url=BASE_URL, max_pages=MAX_PAGES):
    "翻页采集；返回 (记录列表, 采集日志)，结果列表在函数内部，不用全局变量"
    out_list = []                        # 结果列表放在函数局部：每次调用互不污染
    visited = set()
    log = []
    url = start_url
    session = requests.Session()
    session.headers.update(HEADERS)
    page = 0

    while url and page < max_pages:
        page += 1
        norm = url.split("#")[0].rstrip("/") or url
        if norm in visited:
            log.append(f"第{page}页 地址重复，停止：{url}")
            break
        visited.add(norm)
        if urlparse(url).netloc != urlparse(BASE_URL).netloc:
            log.append(f"第{page}页 地址越出教学站，停止：{url}")
            break

        page_soup = fetch(url, session=session)
        page_rows = page_soup.select("div.property")
        if not page_rows:
            log.append(f"第{page}页 房源列表为空，记为异常并停止：{url}")
            break
        records = [parse_row(r) for r in page_rows]
        out_list.extend(records)
        log.append(f"第{page}页 {url} 本页{len(records)}条 累计{len(out_list)}条")

        try:
            url = find_next_url(page_soup, url)
        except StructureError as exc:
            log.append(f"第{page}页 {exc}")
            url = None
            break
        if url:
            time.sleep(SLEEP_SECONDS)    # 至少等待 1 秒再请求下一页

    if page >= max_pages and url:
        log.append(f"达到安全页数上限 {max_pages}，记为中止，不算采集完成")
    return out_list, log


records, crawl_log = crawl()
print("\n".join(crawl_log))
print()
print("采集页数：", len(crawl_log))
print("采集记录数（去重前）：", len(records))

第1页 http://www.bspider.top/anjuke/ 本页60条 累计60条
第2页 http://www.bspider.top/anjuke/?page=2 本页60条 累计120条
第3页 http://www.bspider.top/anjuke/?page=3 本页60条 累计180条
第4页 http://www.bspider.top/anjuke/?page=4 本页60条 累计240条
第5页 http://www.bspider.top/anjuke/?page=5 本页60条 累计300条
第6页 http://www.bspider.top/anjuke/?page=6 本页60条 累计360条
第7页 http://www.bspider.top/anjuke/?page=7 本页60条 累计420条
第8页 http://www.bspider.top/anjuke/?page=8 本页60条 累计480条
第9页 http://www.bspider.top/anjuke/?page=9 本页60条 累计540条
第10页 http://www.bspider.top/anjuke/?page=10 本页60条 累计600条

采集页数： 10
采集记录数（去重前）： 600


## 第二阶段 4　保存并核对 CSV

编码 `utf-8-sig`（Excel 打开不乱码）、`newline=""`（避免写入空行）、表头与 `FIELDS` 顺序一致，
并按 8 个字段完全相同去重。

In [8]:
def records_key(rec):
    "按 8 个字段组成的元组作为去重键"
    return tuple(rec[k] for k in FIELDS)


seen, unique = set(), []
for rec in records:
    key = records_key(rec)
    if key not in seen:
        seen.add(key)
        unique.append(rec)

print("去重前：", len(records))
print("完全相同的重复记录数：", len(records) - len(unique))
print("去重后：", len(unique))

with CSV_PATH.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)                       # 表头
    for rec in unique:
        writer.writerow([rec[k] for k in FIELDS])  # 每条恰好 8 项

print("已写入：", CSV_PATH)

去重前： 600
完全相同的重复记录数： 102
去重后： 498
已写入： data/houseinfo.csv


In [9]:
# 用 DictReader 重新读取，检查行数（表头不计入）
with CSV_PATH.open(encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    rows_back = list(reader)

print("回读数据行数（不含表头）：", len(rows_back))
print("表头：", reader.fieldnames)
print()
print("第 1 条：", [rows_back[0][k] for k in FIELDS])
print("第 61 条（跨页）：", [rows_back[60][k] for k in FIELDS])
print("最后 1 条：", [rows_back[-1][k] for k in FIELDS])

回读数据行数（不含表头）： 498
表头： ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']

第 1 条： ['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']
第 61 条（跨页）： ['万科物业自营 朝中庭 业主诚心出售', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '85.7㎡', '低层(共27层)', '2018年建造', '16914元/㎡']
最后 1 条： ['恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', '恒大照母山山水城', '渝北照母山恒康路9号', '3室2厅1卫', '90.2㎡', '低层(共32层)', '2015年建造', '15530元/㎡']


## 第三阶段　MySQL 数据存储

读取上一步的 `data/houseinfo.csv` 写入 MySQL，**不重新爬取网页**。
连接参数按课程教学环境：`host=mysql`、端口 3306、`student/123456`、`smartpark_lab`、`utf8mb4`。

写入要点：SQL 里用 `%s` 占位、值由 `executemany()` 的第二个参数传入（不做字符串拼接）；
成功 `commit()`，异常 `rollback()`，最后关闭游标与连接。

幂等处理：自增长 `id` 拦不住重复房源，所以先按 8 个字段取出库内已有记录做比对，只插入尚不存在的记录。

In [10]:
import pymysql

DB_CONFIG = {
    "host": "mysql",          # 课程数据库服务名（Notebook 在容器内，不能用 localhost）
    "port": 3306,
    "database": "smartpark_lab",
    "user": "student",
    "password": "123456",
    "charset": "utf8mb4",
    "autocommit": False,      # 由程序显式提交事务
}

conn = pymysql.connect(**DB_CONFIG)
try:
    with conn.cursor() as cur:
        cur.execute("SELECT VERSION(), DATABASE(), CURRENT_USER()")
        print("连接检查：", cur.fetchone())
        cur.execute("SELECT COUNT(*) FROM houseinfo")
        print("入库前条数：", cur.fetchone()[0])
finally:
    conn.close()

连接检查： ('8.4.11', 'smartpark_lab', 'student@%')
入库前条数： 0


In [11]:
INSERT_SQL = (
    "INSERT INTO houseinfo (title, house, address, struct, area, floor, maketime, price) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"     # 8 个占位符对应 8 个字段
)


def load_csv_records(path=CSV_PATH):
    "读 CSV，按 FIELDS 固定顺序整理成元组列表"
    with Path(path).open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != FIELDS:
            raise ValueError(f"CSV 表头与字段顺序不一致：{reader.fieldnames}")
        return [tuple(row[k] for k in FIELDS) for row in reader]


def insert_records(records):
    "把记录写入 houseinfo；返回 (插入条数, 库内原有条数, 写入后总条数)"
    conn = pymysql.connect(**DB_CONFIG)
    cur = None
    try:
        cur = conn.cursor()
        # 1) 取出库内已有记录，按 8 个字段比对，避免重复运行造成重复插入
        cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
        existed = {tuple(r) for r in cur.fetchall()}
        to_insert = [r for r in records if r not in existed]
        print(f"CSV 记录 {len(records)} 条，库内已有 {len(existed)} 条，待插入 {len(to_insert)} 条")

        # 2) 批量写入：executemany 的第二个参数是记录列表，每条 8 个值，顺序与 INSERT 一致
        if to_insert:
            affected = cur.executemany(INSERT_SQL, to_insert)
            print("executemany 受影响行数：", affected)
        else:
            print("没有新记录，跳过写入")

        # 3) 提交事务，另一个 MySQL 终端才能看到新数据
        conn.commit()
        print("commit() 已提交")

        cur.execute("SELECT COUNT(*) FROM houseinfo")
        total = cur.fetchone()[0]
        return len(to_insert), len(existed), total
    except Exception as exc:
        if conn is not None:
            conn.rollback()          # 出错回滚本次未提交的修改
            print("已 rollback()，异常：", type(exc).__name__, exc)
        raise
    finally:
        if cur is not None:
            cur.close()              # 先关游标
        if conn is not None:
            conn.close()             # 再关连接
        print("游标与连接已关闭")


csv_records = load_csv_records()
print("从 CSV 读取：", len(csv_records), "条")
print("第 1 条：", csv_records[0])
print()
inserted, existed_before, total_after = insert_records(csv_records)
print()
print(f"本次新增 {inserted} 条；入库后总条数 {total_after}")

从 CSV 读取：

 498 条
第 1 条： ('万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')



CSV 记录 498 条，库内已有 0 条，待插入 498 条
executemany 受影响行数： 498
commit() 已提交
游标与连接已关闭

本次新增 498 条；入库后总条数 498


In [12]:
# 重跑一次：验证不会重复插入（不清空表）
inserted2, existed2, total2 = insert_records(csv_records)
print()
print(f"第二次运行：新增 {inserted2} 条，总条数 {total2}")
print("重跑是否造成重复插入：", inserted2 != 0)

CSV 记录 498 条，库内已有 498 条，待插入 0 条
没有新记录，跳过写入
commit() 已提交
游标与连接已关闭

第二次运行：新增 0 条，总条数 498
重跑是否造成重复插入： False


In [13]:
# 抽样核对：数据库前 5 条 + 与 CSV 前 3 条逐字段比对
conn = pymysql.connect(**DB_CONFIG)
try:
    with conn.cursor() as cur:
        cur.execute("SELECT id, title, house, address, struct, area, floor, maketime, price "
                    "FROM houseinfo ORDER BY id LIMIT 5")
        print("数据库前 5 条：")
        for r in cur.fetchall():
            print("  ", r)
        cur.execute("SELECT COUNT(*) FROM houseinfo")
        print("入库后总条数：", cur.fetchone()[0])
        cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
        db_rows = [tuple(r) for r in cur.fetchall()]
finally:
    conn.close()

print()
print("CSV 与数据库逐字段一致：", set(csv_records).issubset(set(db_rows)))
print("CSV 条数与库中该批数据条数：", len(csv_records), "/", len(set(csv_records) & set(db_rows)))

数据库前 5 条：
   (1, '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')
   (2, '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡')
   (3, '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '大雅云居山', '渝北鸳鸯龙展路59号', '3室2厅1卫', '88㎡', '低层(共24层)', '2016年建造', '16487元/㎡')
   (4, '中交中央公园 横厅精装大两房 出门都是公园', '中交中央公园悦颂', '渝北中央公园翠丹路4号', '2室2厅1卫', '91㎡', '低层(共25层)', '2016年建造', '20660元/㎡')
   (5, '富力现代广场 豪装大两房 户型方正 看房方便', '富力现代广场', '南岸弹子石弹子石新街59号', '2室2厅1卫', '85.5㎡', '低层(共30层)', '2012年建造', '16969元/㎡')
入库后总条数： 498

CSV 与数据库逐字段一致： True
CSV 条数与库中该批数据条数： 498 / 498
